# Island House — P0 Spike: can OpenSfM auto-align our 360 photos?

**What this does:** takes your 53 stitched 360 photos from the 23 July shoot and runs **OpenSfM** to work out where each one was taken and which way it faces — the auto-alignment that Kuula couldn't do. Then it tells you **GO / MARGINAL / NO-GO**.

**How to use it (nothing to install):**
1. Runtime → *Run all* (or run each cell top to bottom with the ▶ button).
2. When asked, it mounts your Google Drive — put the 53 JPGs in a Drive folder first (see Step 2).
3. Wait for the result at the bottom.

First-time setup takes ~10–15 min (it builds the software), then the run is ~10–30 min. If a cell errors, copy the red error text back to Dominic's desk — SfM installs are finicky and a first-run tweak is normal.


## Step 1 — install OpenSfM (~10–15 min, one time)
This builds the open-source engine. Just run it and wait for **✅ build finished**.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq build-essential cmake git \
  libeigen3-dev libopencv-dev python3-opencv \
  libceres-dev libsuitesparse-dev libgoogle-glog-dev libgflags-dev libboost-python-dev >/dev/null
if [ ! -d /content/OpenSfM ]; then
  git clone --recursive https://github.com/mapillary/OpenSfM.git /content/OpenSfM
fi
cd /content/OpenSfM
pip install -q -r requirements.txt
python3 setup.py build
echo '✅ build finished'


## Step 2 — get your 53 photos in
**Before running this:** in Google Drive, make a folder called **`island-house-360`** in *My Drive* and upload the 53 stitched JPGs into it (`IMG_20260723_..._188.jpg` … `_240.jpg`). Then run this cell — it mounts Drive and copies them into the project.

*(If your folder has a different name/path, edit `IMAGES_DIR` below.)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

IMAGES_DIR = '/content/drive/MyDrive/island-house-360'   # <-- change if your folder is named differently

import os, glob, shutil
from PIL import Image
proj = '/content/project'
os.makedirs(proj + '/images', exist_ok=True)
imgs = sorted(glob.glob(IMAGES_DIR + '/*.jpg') + glob.glob(IMAGES_DIR + '/*.JPG') + glob.glob(IMAGES_DIR + '/*.jpeg'))
assert imgs, 'No images found — check IMAGES_DIR points at your folder of 360 JPGs.'
for p in imgs:
    shutil.copy(p, proj + '/images/')
W, H = Image.open(imgs[0]).size
print(f'Copied {len(imgs)} images. First image is {W}x{H} px.')


## Step 3 — tell it these are 360s (the make-or-break setting)
This writes the config that marks the photos as **spherical (equirectangular)**. Without it, OpenSfM treats them as ordinary photos and fails.

In [ ]:
import json
override = {'all_cameras': {'projection_type': 'spherical', 'width': W, 'height': H}}
open(proj + '/camera_models_overrides.json', 'w').write(json.dumps(override, indent=2))

cfg = (
    'processes: 4\n'
    'feature_process_size: 4096\n'      # downscale from 8K for speed
    'feature_min_frames: 10000\n'
    'matcher_type: FLANN\n'
)
open(proj + '/config.yaml', 'w').write(cfg)
print('Config written. projection_type = spherical,', f'{W}x{H}')
print(open(proj + '/camera_models_overrides.json').read())


## Step 4 — run the alignment (~10–30 min)
This is the actual work: find features, match photos, and reconstruct camera positions. Let it run.

In [ ]:
!/content/OpenSfM/bin/opensfm_run_all /content/project


## Step 5 — the result (GO / MARGINAL / NO-GO)
Reads the reconstruction and prints the verdict.

In [ ]:
import json, glob
rec_path = '/content/project/reconstruction.json'
recs = json.load(open(rec_path))
total = len(glob.glob('/content/project/images/*'))
registered = sum(len(r.get('shots', {})) for r in recs)
print(f'Total photos:        {total}')
print(f'Registered (aligned): {registered}')
print(f'Reconstruction pieces: {len(recs)}  (1 = whole building in one aligned model; more = it broke into fragments)')
for i, r in enumerate(recs):
    print(f'   piece {i+1}: {len(r.get("shots", {}))} photos, {len(r.get("points", {}))} 3D points')

pct = registered / total if total else 0
print('\n================ VERDICT ================')
if pct >= 0.85 and len(recs) == 1:
    print('GO ✅  — clean single aligned model. Build the DIY engine (P1).')
elif pct >= 0.55:
    print('MARGINAL ⚠️  — it works but capture was sparse; a denser reshoot should fix it. Worth continuing.')
else:
    print('NO-GO ❌  — interiors likely too textureless for pure-360 SfM. Fall back to Kuula-by-hand or add markers/depth.')
print('========================================')


### Save the 3D point cloud to eyeball it (optional)
Downloads a `.ply` you can drag into an online viewer (e.g. 3dviewer.net) or MeshLab — the shape should read as rooms and a walk-path if it worked.

In [ ]:
!/content/OpenSfM/bin/opensfm export_ply /content/project
from google.colab import files
try:
    files.download('/content/project/reconstruction.ply')
except Exception as e:
    print('Point cloud is at /content/project/reconstruction.ply — download it from the Files panel.', e)


---
**Report back to the Island House desk** (paste into a Handover row): photos registered / 53 · number of reconstruction pieces · the verdict · a screenshot of the point cloud. If GO, we scope P1 (the viewer + one-wing build).